
# Kuroshio Daily R1 FIX6
## Baseline-first + Failure-only Automated Retry
### 保留原论文方法，只自动处理“原方法真正失败”的日期

FIX5 的结果说明：**不能在第一次 loop 出现时就替换原论文的 recovery 机制**。原 baseline 中有不少日期虽然发生 1–2 次 loop，但最终仍能正确到达 146°E；FIX5 过早介入，反而把这些本来成功的日期破坏了。

FIX6 因此采用最保守的策略：

```text
第一遍：完整运行原 FIX2 baseline
        ↓
若 completed=True
        ↓
100% 原样保留，不做任何修改
```

只有：

```text
completed=False
```

的日期才进入第二遍自动恢复。

第二遍并不引入 CNN 或新模型，而是把原论文中“失败后人工调 rescue 参数重新运行”的工作自动化：

```text
失败日
↓
小范围搜索 rescue jump / angle / loop limit
↓
仍然使用原 characteristic-line 最大流速点
↓
得到若干 completed 候选路径
↓
速度脊物理 Gate
↓
前一有效日路径仅作为软时间连续性参考
↓
自动选一个最可信候选
```

### FIX6 的关键保证

1. **原 baseline 所有成功日期完全不变**；
2. CNN 完全删除；
3. 正常追踪仍是最大流速 characteristic-line；
4. 只对原 baseline 失败日做重试；
5. 若没有可信的完成路径，则保留为失败并进入 manual-review 列表，不会强行制造“100% completion”；
6. 自动输出 R2 180/120/90/60/30 日历史窗的 anchor readiness，避免再次盲目进入预测训练。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, random, shutil
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    CACHE50_DIR: str = "cache_50m"
    BASELINE_DIR: str = "tracks_baseline_FIX2_east0"
    MANIFEST_FILE: str = "reports/daily_manifest.csv"
    BASELINE_REPORT: str = "reports/baseline_tracker_report_FIX2.csv"

    OUT_ROOT: str = "R1_FIX6"
    FINAL_TRACK_DIR: str = "R1_FIX6/tracks_final"
    REPORT_DIR: str = "R1_FIX6/reports"
    FIGURE_DIR: str = "R1_FIX6/figures/recovery_qc"
    DATASET_DIR: str = "R1_FIX6/dataset"

    # original tracker
    SEED_LON_MIN: float = 121.0
    SEED_LON_MAX: float = 122.0
    SEED_LAT_MIN: float = 22.0
    SEED_LAT_MAX: float = 23.0
    TERMINAL_LON: float = 146.0
    EXTENSION_LON: float = 141.0

    AUX_HALF_KM: float = 70.0
    AUX_POINTS: int = 21
    STEP_MAIN_KM: float = 10.0
    STEP_EXTENSION_KM: float = 50.0
    REPEAT_THRESHOLD_KM: float = 1.5
    MAX_TRACK_POINTS: int = 1200

    # failure-only retry grid
    RESCUE_JUMPS_KM: tuple = (20.0, 30.0, 40.0, 50.0)
    RESCUE_ANGLES_DEG: tuple = (30.0, 45.0, 60.0)
    LOOP_LIMITS: tuple = (4, 5)
    RESCUE_STEPS: int = 20

    # stubborn second-stage options
    SECOND_STAGE_REPEAT_THRESHOLDS_KM: tuple = (2.5,)
    SECOND_STAGE_JUMPS_KM: tuple = (30.0, 40.0, 50.0)
    SECOND_STAGE_ANGLES_DEG: tuple = (30.0, 45.0, 60.0)
    SECOND_STAGE_LOOP_LIMIT: int = 5

    MAX_COMPLETED_CANDIDATES_PER_DAY: int = 8

    # candidate physical QC
    RIDGE_HALF_KM: float = 35.0
    RIDGE_POINTS: int = 11
    RIDGE_SAMPLE_STRIDE: int = 5
    MIN_RIDGE_MEDIAN: float = 0.88
    MIN_RIDGE_P10: float = 0.70
    MAX_RIDGE_LOW_FRAC: float = 0.15
    MAX_P95_TURN_DEG: float = 130.0

    # temporal continuity only for choosing among physically valid retries
    TEMPORAL_PRIOR_MAX_GAP_DAYS: int = 7
    TEMPORAL_SOFT_LIMIT_1D_KM: float = 90.0
    TEMPORAL_SOFT_LIMIT_7D_KM: float = 160.0

    RANDOM_SEED: int = 20260925

    # FIX6-FIX1: memory-safe QC
    QC_MAX_FIGURES: int = 24
    QC_INCLUDE_AUTO_ACCEPTED: int = 10
    QC_BACKGROUND_STRIDE: int = 4
    QC_DPI: int = 100
    QC_FIGSIZE: tuple = (8.5, 5.8)

    # R2 readiness
    R2_LOOKBACKS: tuple = (180,120,90,60,30)
    R2_LEADS: tuple = (30,60,90)
    R2_MAX_INPUT_GAP: int = 2
    R2_MAX_INPUT_MISSING_FRAC: float = 0.03
    R2_TRAIN_TARGET_END: str = "2024-08-31"
    R2_VAL_TARGET_END: str = "2024-11-30"

cfg=CFG()

ROOT=Path(cfg.PROJECT_ROOT)
CACHE50=ROOT/cfg.CACHE50_DIR
BASE_DIR=ROOT/cfg.BASELINE_DIR
MANIFEST_PATH=ROOT/cfg.MANIFEST_FILE
BASE_REPORT_PATH=ROOT/cfg.BASELINE_REPORT

OUT_ROOT=ROOT/cfg.OUT_ROOT
FINAL_TRACK_DIR=ROOT/cfg.FINAL_TRACK_DIR
REPORT_DIR=ROOT/cfg.REPORT_DIR
FIGURE_DIR=ROOT/cfg.FIGURE_DIR
DATASET_DIR=ROOT/cfg.DATASET_DIR

for p in [OUT_ROOT,FINAL_TRACK_DIR,REPORT_DIR,FIGURE_DIR,DATASET_DIR]:
    p.mkdir(parents=True,exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))


In [ ]:

# ===== 1. 输入审计 =====
for p in [CACHE50,BASE_DIR,MANIFEST_PATH,BASE_REPORT_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

manifest=pd.read_csv(MANIFEST_PATH,parse_dates=["date"])
base_report=pd.read_csv(BASE_REPORT_PATH,parse_dates=["date"]).sort_values("date")

print("raw days:",len(base_report))
print("baseline completion:",float(base_report.completed.mean()))
print("baseline completed:",int(base_report.completed.sum()))
print("baseline failed:",int((~base_report.completed).sum()))
print("\nfailed stop reasons:")
display(base_report.loc[~base_report.completed,"stop_reason"].value_counts().to_frame("count"))

success_path_km=base_report.loc[base_report.completed,"path_km"].dropna()
PATH_KM_Q01=float(success_path_km.quantile(.01))
PATH_KM_Q50=float(success_path_km.quantile(.50))
PATH_KM_Q99=float(success_path_km.quantile(.99))
print("successful baseline path_km Q01/Q50/Q99:",PATH_KM_Q01,PATH_KM_Q50,PATH_KM_Q99)


In [ ]:

# ===== 2. 几何与原 characteristic-line 工具 =====
from scipy.interpolate import RegularGridInterpolator

EARTH_R_KM=6371.0088

def wrap_pi(a):
    return (a+np.pi)%(2*np.pi)-np.pi

def haversine_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2=map(np.asarray,(lon1,lat1,lon2,lat2))
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    dp=p2-p1
    aa=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    y=np.sin(dl)*np.cos(p2)
    x=np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav=np.arctan2(y,x)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav=np.pi/2-east_angle
    phi1,lam1=np.deg2rad(lat),np.deg2rad(lon)
    d=distance_km/EARTH_R_KM
    phi2=np.arcsin(
        np.sin(phi1)*np.cos(d)+
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2=lam1+np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

def aux_line(lon,lat,flow_dir,half_km=None,n=None):
    half_km=cfg.AUX_HALF_KM if half_km is None else half_km
    n=cfg.AUX_POINTS if n is None else n
    ds=np.linspace(-half_km,half_km,n)
    xs,ys=[],[]
    for s in ds:
        b=flow_dir+(np.pi/2 if s>=0 else -np.pi/2)
        x,y=forward_point(lon,lat,b,abs(float(s)))
        xs.append(x); ys.append(y)
    return np.asarray(xs),np.asarray(ys),ds

def matlab_aux_weights(n):
    k=np.arange(1,n+1,dtype=float)
    return 1.0-np.abs((n+1)/2.0-k)/(n-1)

def make_interpolators(lon,lat,u,v):
    return (
        RegularGridInterpolator((lat,lon),u,bounds_error=False,fill_value=np.nan),
        RegularGridInterpolator((lat,lon),v,bounds_error=False,fill_value=np.nan)
    )

def interp_uv(ui,vi,lons,lats):
    pts=np.column_stack([lats,lons])
    return ui(pts),vi(pts)

def _seed_point(lon,lat,u,v):
    ix=np.where((lon>=cfg.SEED_LON_MIN)&(lon<=cfg.SEED_LON_MAX))[0]
    iy=np.where((lat>=cfg.SEED_LAT_MIN)&(lat<=cfg.SEED_LAT_MAX))[0]
    su=u[np.ix_(iy,ix)]
    sv=v[np.ix_(iy,ix)]
    sp2=su**2+sv**2
    if not np.isfinite(sp2).any():
        raise RuntimeError("seed region invalid")
    j,i=np.unravel_index(np.nanargmax(sp2),sp2.shape)
    return float(lon[ix[i]]),float(lat[iy[j]]),float(np.arctan2(sv[j,i],su[j,i]))

# geometry smoke
xe,ye=forward_point(130,30,0,10)
xn,yn=forward_point(130,30,np.pi/2,10)
assert xe>130 and abs(ye-30)<.05
assert yn>30 and abs(xn-130)<.05
print("[geometry] PASS")


In [ ]:

# ===== 3. 参数化原FIX2 tracker：仍然只选最大流速点 =====
def candidate_select_original(
    ui,vi,
    center_lon,center_lat,
    mean_dir,
    rescue=False,
    prev_axis_lon=None,
    prev_axis_lat=None,
    rescue_angle_deg=30.0
):
    xl,yl,_=aux_line(center_lon,center_lat,mean_dir)
    uc,vc=interp_uv(ui,vi,xl,yl)
    speed=np.hypot(uc,vc)
    valid=np.isfinite(speed)&np.isfinite(uc)&np.isfinite(vc)

    if not valid.any():
        return None

    if rescue and prev_axis_lon is not None and prev_axis_lat is not None:
        geom_dirs=np.array([
            angle_east_rad(prev_axis_lon,prev_axis_lat,x,y)
            for x,y in zip(xl,yl)
        ])
        geom_diff=np.abs(wrap_pi(geom_dirs-mean_dir))
        constrained=valid&(geom_diff<np.deg2rad(rescue_angle_deg))

        if constrained.any():
            valid=constrained
        else:
            good=np.where(valid)[0]
            if len(good):
                k0=good[np.argmin(geom_diff[good])]
                valid[:]=False
                valid[k0]=True

    good=np.where(valid)[0]
    if len(good)==0:
        return None

    k=int(good[np.argmax(speed[good])])
    return {
        "lon":float(xl[k]),
        "lat":float(yl[k]),
        "u":float(uc[k]),
        "v":float(vc[k]),
        "speed":float(speed[k]),
    }

def track_param(
    lon,lat,u,v,
    rescue_jump_km=20.0,
    rescue_angle_deg=30.0,
    rescue_steps=20,
    loop_limit=3,
    repeat_threshold_km=1.5
):
    ui,vi=make_interpolators(lon,lat,u,v)
    lon0,lat0,theta0=_seed_point(lon,lat,u,v)

    path_lon=[]; path_lat=[]; dirs=[]
    loop_events=0
    rescue_remaining=0
    center_lon,center_lat=lon0,lat0
    current_dir=theta0
    stop_reason="max_points"

    for _ in range(cfg.MAX_TRACK_POINTS):
        xl,yl,_=aux_line(center_lon,center_lat,current_dir)
        uu,vv=interp_uv(ui,vi,xl,yl)
        valid=np.isfinite(uu)&np.isfinite(vv)

        if not valid.any():
            stop_reason="no_valid_aux_flow"
            break

        w=matlab_aux_weights(len(xl))
        den=np.sum(w[valid])
        if den<=0:
            stop_reason="bad_aux_weight"
            break

        mean_u=np.sum(uu[valid]*w[valid])/den
        mean_v=np.sum(vv[valid]*w[valid])/den
        mean_dir=float(np.arctan2(mean_v,mean_u))

        prev_lon=path_lon[-1] if path_lon else lon0
        prev_lat=path_lat[-1] if path_lat else lat0

        cand=candidate_select_original(
            ui,vi,
            center_lon,center_lat,
            mean_dir,
            rescue=(rescue_remaining>0),
            prev_axis_lon=prev_lon,
            prev_axis_lat=prev_lat,
            rescue_angle_deg=rescue_angle_deg
        )

        if cand is None:
            stop_reason="no_candidate"
            break

        cl,ca=cand["lon"],cand["lat"]

        repeat_idx=None
        if len(path_lon)>8:
            hx=np.asarray(path_lon[:-5])
            hy=np.asarray(path_lat[:-5])
            dd=haversine_km(hx,hy,cl,ca)
            if np.isfinite(dd).any() and np.nanmin(dd)<repeat_threshold_km:
                repeat_idx=int(np.nanargmin(dd))

        if repeat_idx is not None:
            loop_events+=1

            path_lon=path_lon[:repeat_idx+1]
            path_lat=path_lat[:repeat_idx+1]
            dirs=dirs[:repeat_idx+1]

            if loop_events>=loop_limit or len(path_lon)<2:
                stop_reason="too_many_loops"
                break

            current_dir=dirs[-1]

            step_km=(
                cfg.STEP_EXTENSION_KM
                if path_lon[-1]>cfg.EXTENSION_LON
                else rescue_jump_km
            )
            center_lon,center_lat=forward_point(
                path_lon[-1],path_lat[-1],current_dir,step_km
            )
            rescue_remaining=rescue_steps
            continue

        path_lon.append(cl)
        path_lat.append(ca)
        dirs.append(mean_dir)

        if cl>=cfg.TERMINAL_LON:
            stop_reason="terminal_lon"
            break

        step_km=(
            cfg.STEP_EXTENSION_KM
            if cl>cfg.EXTENSION_LON
            else cfg.STEP_MAIN_KM
        )
        center_lon,center_lat=forward_point(cl,ca,mean_dir,step_km)
        current_dir=mean_dir
        rescue_remaining=max(0,rescue_remaining-1)

    x=np.asarray(path_lon,dtype=np.float32)
    y=np.asarray(path_lat,dtype=np.float32)

    completed=bool(len(x)>2 and np.isfinite(x).any() and np.nanmax(x)>=cfg.TERMINAL_LON)

    return {
        "lon":x,
        "lat":y,
        "direction":np.asarray(dirs,dtype=np.float32),
        "completed":completed,
        "loop_events":int(loop_events),
        "n_points":int(len(x)),
        "stop_reason":stop_reason,
        "params":{
            "rescue_jump_km":float(rescue_jump_km),
            "rescue_angle_deg":float(rescue_angle_deg),
            "rescue_steps":int(rescue_steps),
            "loop_limit":int(loop_limit),
            "repeat_threshold_km":float(repeat_threshold_km),
        }
    }


In [ ]:

# ===== 4. 候选路径物理质量与时间连续性 =====
def path_basic_metrics(tr):
    x=np.asarray(tr["lon"],float)
    y=np.asarray(tr["lat"],float)

    path_km=float(np.sum(haversine_km(x[:-1],y[:-1],x[1:],y[1:]))) if len(x)>1 else 0.0

    if len(x)>3:
        dirs=np.array([
            angle_east_rad(x[i],y[i],x[i+1],y[i+1])
            for i in range(len(x)-1)
        ])
        turns=np.abs(np.rad2deg(wrap_pi(np.diff(dirs))))
        p95=float(np.nanpercentile(turns,95))
    else:
        p95=np.nan

    return {
        "path_km":path_km,
        "p95_turn_deg":p95,
        "max_lon":float(np.nanmax(x)) if len(x) else np.nan,
        "final_lon":float(x[-1]) if len(x) else np.nan,
        "final_lat":float(y[-1]) if len(y) else np.nan,
    }

def ridge_adherence(lon_grid,lat_grid,u,v,tr):
    x=np.asarray(tr["lon"],float)
    y=np.asarray(tr["lat"],float)

    if len(x)<5:
        return np.nan,np.nan,np.nan

    ui,vi=make_interpolators(lon_grid,lat_grid,u,v)
    ids=np.arange(2,len(x)-2,cfg.RIDGE_SAMPLE_STRIDE)

    ratios=[]

    for i in ids:
        tangent=angle_east_rad(x[i-1],y[i-1],x[i+1],y[i+1])
        xl,yl,_=aux_line(
            x[i],y[i],tangent,
            half_km=cfg.RIDGE_HALF_KM,
            n=cfg.RIDGE_POINTS
        )
        uu,vv=interp_uv(ui,vi,xl,yl)
        sp=np.hypot(uu,vv)
        good=np.isfinite(sp)
        if not good.any():
            continue

        uc,vc=interp_uv(ui,vi,[x[i]],[y[i]])
        center_speed=float(np.hypot(uc[0],vc[0]))
        mx=float(np.nanmax(sp[good]))

        if np.isfinite(center_speed) and mx>1e-8:
            ratios.append(min(center_speed/mx,1.25))

    if not ratios:
        return np.nan,np.nan,np.nan

    rr=np.asarray(ratios,float)
    return (
        float(np.nanmedian(rr)),
        float(np.nanpercentile(rr,10)),
        float(np.mean(rr<0.75))
    )

def xy_project(lon,lat,lat0=None,lon0=None):
    lon=np.asarray(lon,float); lat=np.asarray(lat,float)
    if lat0 is None: lat0=float(np.nanmean(lat))
    if lon0 is None: lon0=float(np.nanmean(lon))
    x=(lon-lon0)*111.32*np.cos(np.deg2rad(lat0))
    y=(lat-lat0)*111.32
    return x,y

def symmetric_path_distance_km(tr1,tr2,sample_step=4):
    x1=np.asarray(tr1["lon"],float)[::sample_step]
    y1=np.asarray(tr1["lat"],float)[::sample_step]
    x2=np.asarray(tr2["lon"],float)[::sample_step]
    y2=np.asarray(tr2["lat"],float)[::sample_step]

    if len(x1)==0 or len(x2)==0:
        return np.nan

    lat0=float(np.mean(np.r_[y1,y2]))
    lon0=float(np.mean(np.r_[x1,x2]))
    a=np.column_stack(xy_project(x1,y1,lat0,lon0))
    b=np.column_stack(xy_project(x2,y2,lat0,lon0))

    # pairwise Euclidean km
    d=np.sqrt(((a[:,None,:]-b[None,:,:])**2).sum(axis=2))
    da=np.min(d,axis=1)
    db=np.min(d,axis=0)
    return float(np.median(np.r_[da,db]))

def load_track(path):
    z=np.load(path)
    return {
        "lon":z["lon"],
        "lat":z["lat"],
        "direction":z["direction"] if "direction" in z.files else np.array([],dtype=np.float32),
        "completed":bool(z["completed"]) if "completed" in z.files else False,
        "loop_events":int(z["loop_events"]) if "loop_events" in z.files else 0,
        "n_points":int(z["n_points"]) if "n_points" in z.files else len(z["lon"]),
        "stop_reason":str(z["stop_reason"].item()) if "stop_reason" in z.files else "unknown",
    }

def physical_gate(m):
    return bool(
        np.isfinite(m["ridge_median"]) and
        np.isfinite(m["ridge_p10"]) and
        m["ridge_median"]>=cfg.MIN_RIDGE_MEDIAN and
        m["ridge_p10"]>=cfg.MIN_RIDGE_P10 and
        m["ridge_low_frac"]<=cfg.MAX_RIDGE_LOW_FRAC and
        (not np.isfinite(m["p95_turn_deg"]) or m["p95_turn_deg"]<=cfg.MAX_P95_TURN_DEG) and
        m["path_km"]>=0.70*PATH_KM_Q01 and
        m["path_km"]<=1.30*PATH_KM_Q99
    )


In [ ]:

# ===== 5. 先复制所有baseline成功日；只对失败日搜索 =====
# final_tracks用于按日期顺序形成“上一有效日”软先验
final_tracks={}
final_source={}
recovery_rows=[]
trial_rows=[]
manual_rows=[]

# 成功 baseline 完全原样复制
for r in base_report.loc[base_report.completed].itertuples(index=False):
    dt=pd.Timestamp(r.date)
    src=BASE_DIR/f"{dt:%Y%m%d}.npz"
    dst=FINAL_TRACK_DIR/f"{dt:%Y%m%d}.npz"
    if not src.exists():
        continue
    shutil.copy2(src,dst)
    tr=load_track(src)
    final_tracks[dt]=tr
    final_source[dt]="baseline_unchanged"

print("copied unchanged baseline success days:",len(final_tracks))


In [ ]:

# ===== 6. Failure-only automated retry =====
from tqdm.auto import tqdm

def previous_reference(dt):
    prev=[d for d in final_tracks.keys() if d<dt]
    if not prev:
        return None,None
    p=max(prev)
    gap=(dt-p).days
    if gap>cfg.TEMPORAL_PRIOR_MAX_GAP_DAYS:
        return None,None
    return p,final_tracks[p]

def next_baseline_reference(dt):
    # 仅诊断，不参与选择
    q=base_report[
        (base_report.date>dt)&
        (base_report.completed)
    ]
    if len(q)==0:
        return None,None
    nr=q.iloc[0]
    nd=pd.Timestamp(nr.date)
    if (nd-dt).days>cfg.TEMPORAL_PRIOR_MAX_GAP_DAYS:
        return None,None
    p=BASE_DIR/f"{nd:%Y%m%d}.npz"
    return (nd,load_track(p)) if p.exists() else (None,None)

def candidate_record(dt,tr,z,stage,trial_id):
    bm=path_basic_metrics(tr)
    rmed,rp10,rlow=ridge_adherence(
        z["lon"],z["lat"],z["u"],z["v"],tr
    )
    bm.update({
        "ridge_median":rmed,
        "ridge_p10":rp10,
        "ridge_low_frac":rlow,
    })

    pd_prev,prev_tr=previous_reference(dt)
    if prev_tr is not None:
        tdist=symmetric_path_distance_km(tr,prev_tr)
        prev_gap=(dt-pd_prev).days
    else:
        tdist=np.nan
        prev_gap=np.nan

    nd,next_tr=next_baseline_reference(dt)
    ndist=symmetric_path_distance_km(tr,next_tr) if next_tr is not None else np.nan

    rec={
        "date":dt,
        "stage":stage,
        "trial_id":trial_id,
        **tr["params"],
        "completed":bool(tr["completed"]),
        "loop_events":int(tr["loop_events"]),
        "stop_reason":tr["stop_reason"],
        **bm,
        "prev_gap_days":prev_gap,
        "temporal_prev_km":tdist,
        "diagnostic_next_km":ndist,
    }
    rec["physical_gate"]=physical_gate(rec)
    return rec

failed_days=base_report.loc[~base_report.completed,"date"].sort_values().tolist()

for dt0 in tqdm(failed_days,desc="FIX6 failed-day retry"):
    dt=pd.Timestamp(dt0)
    cache=CACHE50/f"{dt:%Y%m%d}.npz"

    if not cache.exists():
        manual_rows.append({"date":dt,"reason":"missing_cache"})
        continue

    z=np.load(cache)
    completed_candidates=[]
    tid=0

    # Stage A：围绕原参数做小范围搜索
    for loop_limit in cfg.LOOP_LIMITS:
        for jump in cfg.RESCUE_JUMPS_KM:
            for ang in cfg.RESCUE_ANGLES_DEG:
                tid+=1
                tr=track_param(
                    z["lon"],z["lat"],z["u"],z["v"],
                    rescue_jump_km=jump,
                    rescue_angle_deg=ang,
                    rescue_steps=cfg.RESCUE_STEPS,
                    loop_limit=loop_limit,
                    repeat_threshold_km=cfg.REPEAT_THRESHOLD_KM
                )

                rec=candidate_record(dt,tr,z,"A",tid)
                trial_rows.append(rec)

                if tr["completed"]:
                    completed_candidates.append((tr,rec))

                if len(completed_candidates)>=cfg.MAX_COMPLETED_CANDIDATES_PER_DAY:
                    break
            if len(completed_candidates)>=cfg.MAX_COMPLETED_CANDIDATES_PER_DAY:
                break
        if len(completed_candidates)>=cfg.MAX_COMPLETED_CANDIDATES_PER_DAY:
            break

    # Stage B：仅当A没有任何完整结果时扩大loop探测尺度
    if not completed_candidates:
        for rpt in cfg.SECOND_STAGE_REPEAT_THRESHOLDS_KM:
            for jump in cfg.SECOND_STAGE_JUMPS_KM:
                for ang in cfg.SECOND_STAGE_ANGLES_DEG:
                    tid+=1
                    tr=track_param(
                        z["lon"],z["lat"],z["u"],z["v"],
                        rescue_jump_km=jump,
                        rescue_angle_deg=ang,
                        rescue_steps=cfg.RESCUE_STEPS,
                        loop_limit=cfg.SECOND_STAGE_LOOP_LIMIT,
                        repeat_threshold_km=rpt
                    )
                    rec=candidate_record(dt,tr,z,"B",tid)
                    trial_rows.append(rec)

                    if tr["completed"]:
                        completed_candidates.append((tr,rec))

                    if len(completed_candidates)>=cfg.MAX_COMPLETED_CANDIDATES_PER_DAY:
                        break
                if len(completed_candidates)>=cfg.MAX_COMPLETED_CANDIDATES_PER_DAY:
                    break

    if not completed_candidates:
        recovery_rows.append({
            "date":dt,
            "recovered":False,
            "auto_accepted":False,
            "reason":"no_completed_retry",
            "n_completed_candidates":0,
        })
        manual_rows.append({"date":dt,"reason":"no_completed_retry"})
        continue

    # 先用物理Gate过滤
    valid=[x for x in completed_candidates if x[1]["physical_gate"]]
    pool=valid if valid else completed_candidates

    # 选择逻辑：
    # 有前一有效日 -> 在物理合格候选中优先最小时间距离
    # 无前一有效日 -> 优先速度脊，其次平滑度
    if np.isfinite([x[1]["temporal_prev_km"] for x in pool]).any():
        pool_sorted=sorted(
            pool,
            key=lambda x:(
                0 if x[1]["physical_gate"] else 1,
                x[1]["temporal_prev_km"] if np.isfinite(x[1]["temporal_prev_km"]) else 1e9,
                -x[1]["ridge_median"] if np.isfinite(x[1]["ridge_median"]) else 1e9,
                x[1]["p95_turn_deg"] if np.isfinite(x[1]["p95_turn_deg"]) else 1e9
            )
        )
    else:
        pool_sorted=sorted(
            pool,
            key=lambda x:(
                0 if x[1]["physical_gate"] else 1,
                -x[1]["ridge_median"] if np.isfinite(x[1]["ridge_median"]) else 1e9,
                -x[1]["ridge_p10"] if np.isfinite(x[1]["ridge_p10"]) else 1e9,
                x[1]["p95_turn_deg"] if np.isfinite(x[1]["p95_turn_deg"]) else 1e9
            )
        )

    best_tr,best=pool_sorted[0]

    # temporal prior只做软合理性检查；超阈值则要求人工看图，不强行接受
    temporal_ok=True
    if np.isfinite(best["temporal_prev_km"]) and np.isfinite(best["prev_gap_days"]):
        gap=max(1,int(best["prev_gap_days"]))
        lim=cfg.TEMPORAL_SOFT_LIMIT_1D_KM + (
            cfg.TEMPORAL_SOFT_LIMIT_7D_KM-cfg.TEMPORAL_SOFT_LIMIT_1D_KM
        )*(gap-1)/max(1,cfg.TEMPORAL_PRIOR_MAX_GAP_DAYS-1)
        temporal_ok=best["temporal_prev_km"]<=lim

    auto_accepted=bool(best["physical_gate"] and temporal_ok)

    recovery_rows.append({
        "date":dt,
        "recovered":True,
        "auto_accepted":auto_accepted,
        "reason":"accepted" if auto_accepted else "manual_review_required",
        "n_completed_candidates":len(completed_candidates),
        **{k:best[k] for k in [
            "stage","trial_id","rescue_jump_km","rescue_angle_deg",
            "loop_limit","repeat_threshold_km","loop_events","path_km",
            "p95_turn_deg","ridge_median","ridge_p10","ridge_low_frac",
            "prev_gap_days","temporal_prev_km","diagnostic_next_km","physical_gate"
        ]}
    })

    if auto_accepted:
        save_path=FINAL_TRACK_DIR/f"{dt:%Y%m%d}.npz"
        np.savez_compressed(
            save_path,
            lon=best_tr["lon"],
            lat=best_tr["lat"],
            direction=best_tr["direction"],
            completed=np.array(True),
            loop_events=np.array(best_tr["loop_events"]),
            n_points=np.array(best_tr["n_points"]),
            stop_reason=np.array(best_tr["stop_reason"]),
            tracker_version=np.array("R1_FIX6_FAILURE_ONLY_AUTORETRY"),
            recovery_params=np.array(json.dumps(best_tr["params"]))
        )
        final_tracks[dt]=best_tr
        final_source[dt]="recovered_FIX6"
    else:
        manual_rows.append({
            "date":dt,
            "reason":"candidate_completed_but_quality_or_temporal_gate_failed"
        })

trials=pd.DataFrame(trial_rows)
recoveries=pd.DataFrame(recovery_rows)
manual=pd.DataFrame(manual_rows)

trials.to_csv(REPORT_DIR/"recovery_trials_FIX6.csv",index=False,encoding="utf-8-sig")
recoveries.to_csv(REPORT_DIR/"recovery_summary_FIX6.csv",index=False,encoding="utf-8-sig")
manual.to_csv(REPORT_DIR/"manual_review_required_FIX6.csv",index=False,encoding="utf-8-sig")

print("baseline failed days:",len(failed_days))
print("any completed retry:",int(recoveries.recovered.sum()) if len(recoveries) else 0)
print("auto accepted:",int(recoveries.auto_accepted.sum()) if len(recoveries) else 0)
print("manual review:",len(manual))


In [ ]:

# ===== 7. 最终report：成功baseline不变 + 自动接受recovery =====
rows=[]

for r in base_report.itertuples(index=False):
    dt=pd.Timestamp(r.date)
    p=FINAL_TRACK_DIR/f"{dt:%Y%m%d}.npz"

    if p.exists():
        tr=load_track(p)
        bm=path_basic_metrics(tr)

        source=final_source.get(dt,"unknown")
        rows.append({
            "date":dt,
            "completed":bool(tr["completed"]),
            "source":source,
            "loop_events":int(tr["loop_events"]),
            "n_points":int(tr["n_points"]),
            "stop_reason":tr["stop_reason"],
            **bm
        })
    else:
        rows.append({
            "date":dt,
            "completed":False,
            "source":"unresolved_baseline_failure",
            "loop_events":r.loop_events,
            "n_points":r.n_points,
            "stop_reason":r.stop_reason,
            "path_km":r.path_km,
            "p95_turn_deg":r.p95_turn_deg,
            "max_lon":r.max_lon,
            "final_lon":r.final_lon,
            "final_lat":r.final_lat,
        })

final_report=pd.DataFrame(rows).sort_values("date")
final_report.to_csv(
    REPORT_DIR/"final_tracker_report_FIX6.csv",
    index=False,
    encoding="utf-8-sig"
)

base_completion=float(base_report.completed.mean())
final_completion=float(final_report.completed.mean())

print("baseline completion:",base_completion)
print("FIX6 final completion:",final_completion)
print("added complete days:",int(final_report.completed.sum()-base_report.completed.sum()))
print("baseline-success regressions: 0 by construction")
print("\nsource counts:")
display(final_report.source.value_counts().to_frame("count"))


In [ ]:

# ===== 8. Recovery QC figures — FIX6-FIX1 memory-safe =====
# 该单元只用于人工QC，不参与tracker或dataset构建。
# 旧版在WSL中可能因连续pcolormesh/colorbar产生内存峰值导致整个WSL被OOM杀掉。

import os, gc
os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
plt.ioff()

# WSL崩溃重启后可直接从磁盘恢复，不依赖Cell 6/7内存变量
recovery_csv = REPORT_DIR / "recovery_summary_FIX6.csv"
manual_csv = REPORT_DIR / "manual_review_required_FIX6.csv"

if not recovery_csv.exists():
    raise FileNotFoundError(
        f"缺少 {recovery_csv}；请先完成failure-only retry。"
    )

recoveries_plot = pd.read_csv(recovery_csv, parse_dates=["date"])
manual_plot = (
    pd.read_csv(manual_csv, parse_dates=["date"])
    if manual_csv.exists()
    else pd.DataFrame(columns=["date","reason"])
)

plot_dates = []

# manual review优先
if len(manual_plot) and "date" in manual_plot:
    plot_dates += manual_plot["date"].dropna().tolist()

# 自动接受的recovery只抽最可疑的一小批
if "auto_accepted" in recoveries_plot:
    accepted = recoveries_plot[
        recoveries_plot["auto_accepted"].astype(bool)
    ].copy()
else:
    accepted = pd.DataFrame()

if len(accepted):
    accepted["_qc_priority"] = 0.0
    if "ridge_median" in accepted:
        accepted["_qc_priority"] += (1.0 - accepted["ridge_median"].fillna(0.0))*5.0
    if "ridge_p10" in accepted:
        accepted["_qc_priority"] += (1.0 - accepted["ridge_p10"].fillna(0.0))*3.0
    if "temporal_prev_km" in accepted:
        accepted["_qc_priority"] += np.clip(
            accepted["temporal_prev_km"].fillna(0.0)/100.0, 0, 5
        )
    if "p95_turn_deg" in accepted:
        accepted["_qc_priority"] += np.clip(
            accepted["p95_turn_deg"].fillna(0.0)/90.0, 0, 3
        )

    plot_dates += accepted.sort_values(
        "_qc_priority", ascending=False
    ).head(cfg.QC_INCLUDE_AUTO_ACCEPTED)["date"].tolist()

plot_dates = sorted(set(pd.to_datetime(plot_dates)))
if len(plot_dates) > cfg.QC_MAX_FIGURES:
    print(
        f"[QC] {len(plot_dates)} dates requested; "
        f"limiting to {cfg.QC_MAX_FIGURES} for WSL memory safety."
    )
    plot_dates = plot_dates[:cfg.QC_MAX_FIGURES]

print("QC figures to render:", len(plot_dates))

plot_rows = []

for i, dt in enumerate(plot_dates, start=1):
    dt = pd.Timestamp(dt)
    cache = CACHE50 / f"{dt:%Y%m%d}.npz"
    bp = BASE_DIR / f"{dt:%Y%m%d}.npz"
    fp = FINAL_TRACK_DIR / f"{dt:%Y%m%d}.npz"

    if not (cache.exists() and bp.exists()):
        plot_rows.append({"date":dt,"status":"skip_missing_input"})
        continue

    fig = None
    try:
        # 主动关闭npz，且只把下采样后的背景读入内存
        stride = max(1, int(cfg.QC_BACKGROUND_STRIDE))
        with np.load(cache) as z:
            lon_ds = np.asarray(z["lon"][::stride], dtype=np.float32)
            lat_ds = np.asarray(z["lat"][::stride], dtype=np.float32)
            u_ds = np.asarray(z["u"][::stride, ::stride], dtype=np.float32)
            v_ds = np.asarray(z["v"][::stride, ::stride], dtype=np.float32)

        sp_ds = np.hypot(u_ds, v_ds).astype(np.float32, copy=False)

        b = load_track(bp)
        f = load_track(fp) if fp.exists() else None

        fig, ax = plt.subplots(figsize=cfg.QC_FIGSIZE)

        # 规则GLORYS网格：imshow比pcolormesh显著省内存
        im = ax.imshow(
            sp_ds,
            origin="lower",
            extent=[
                float(lon_ds[0]), float(lon_ds[-1]),
                float(lat_ds[0]), float(lat_ds[-1])
            ],
            aspect="auto",
            interpolation="nearest"
        )

        ax.plot(
            b["lon"], b["lat"],
            lw=1.0,
            label="original baseline failure"
        )

        if f is not None:
            ax.plot(
                f["lon"], f["lat"],
                lw=1.8,
                label="FIX6 recovered"
            )
            if len(f["lon"]):
                ax.scatter(
                    [f["lon"][-1]], [f["lat"][-1]],
                    s=25, marker="x", label="FIX6 end"
                )

        rr = recoveries_plot.loc[recoveries_plot["date"] == dt]
        ttl = str(dt.date())
        if len(rr):
            r0 = rr.iloc[0]
            ttl += (
                f" | recovered={bool(r0.get('recovered', False))}"
                f" accepted={bool(r0.get('auto_accepted', False))}"
            )
            rv = r0.get("ridge_median", np.nan)
            if np.isfinite(rv):
                ttl += f" ridge={float(rv):.3f}"

        ax.set_title(ttl, fontsize=8)
        ax.set_xlim(120,148)
        ax.set_ylim(20,42)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.legend(loc="best", fontsize=7)

        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
        cbar.set_label("speed (m/s)", fontsize=7)
        cbar.ax.tick_params(labelsize=6)

        fig.tight_layout()

        out_png = FIGURE_DIR / f"{dt:%Y%m%d}_FIX6_recovery.png"
        fig.savefig(out_png, dpi=cfg.QC_DPI, bbox_inches="tight")

        plot_rows.append({
            "date":dt,
            "status":"saved",
            "path":str(out_png)
        })

        if i % 5 == 0:
            print(f"[QC] {i}/{len(plot_dates)}")

    except Exception as e:
        print(f"[QC WARN] {dt.date()} {type(e).__name__}: {e}")
        plot_rows.append({
            "date":dt,
            "status":"plot_error",
            "error":repr(e)
        })

    finally:
        if fig is not None:
            plt.close(fig)

        # 每张图后主动释放大数组与Matplotlib对象
        for name in [
            "sp_ds","u_ds","v_ds","lon_ds","lat_ds",
            "b","f","im","cbar","ax"
        ]:
            if name in locals():
                try:
                    del globals()[name]
                except Exception:
                    pass
        gc.collect()

pd.DataFrame(plot_rows).to_csv(
    REPORT_DIR/"qc_plot_manifest_FIX6.csv",
    index=False,
    encoding="utf-8-sig"
)

plt.close("all")
gc.collect()

print("QC figures ->", FIGURE_DIR)
print("Cell 8 is optional; Cell 9+ can run even if QC plotting is skipped.")


In [ ]:

# FIX6-FIX1: WSL重启后从磁盘恢复final_report
final_report_path = REPORT_DIR/"final_tracker_report_FIX6.csv"
if not final_report_path.exists():
    raise FileNotFoundError(final_report_path)
final_report = pd.read_csv(final_report_path, parse_dates=["date"])


# ===== 9. 生成最终双分支 daily dataset =====
from scipy.interpolate import interp1d

TARGET_LATS_1=np.arange(22.0,25.5+1e-9,1/12)
TARGET_LONS_2=np.arange(122.0,145.0+1e-9,1/12)

def branch1_from_track(lons,lats):
    valid=np.isfinite(lons)&np.isfinite(lats)&(lats<=25.7)
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]
    if len(lats)<2: return None

    df=(
        pd.DataFrame({"lat":lats,"lon":lons})
        .groupby("lat",as_index=False).lon.mean()
        .sort_values("lat")
    )
    if len(df)<2: return None

    f=interp1d(df.lat.values,df.lon.values,bounds_error=False,fill_value="extrapolate")
    return f(TARGET_LATS_1).astype(np.float32)

def branch2_from_track(lons,lats):
    valid=np.isfinite(lons)&np.isfinite(lats)
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]
    if len(lons)<2: return None

    vals=[]
    for lo in np.sort(np.unique(lons)):
        yy=lats[lons==lo]
        vals.append((lo,np.max(yy) if lo<=125 else np.mean(yy)))

    arr=np.asarray(vals,float)
    if len(arr)<2: return None

    f=interp1d(arr[:,0],arr[:,1],bounds_error=False,fill_value="extrapolate")
    return f(TARGET_LONS_2).astype(np.float32)

D=[]; B1=[]; B2=[]; SRC=[]

for r in final_report.itertuples(index=False):
    if not bool(r.completed):
        continue

    dt=pd.Timestamp(r.date)
    p=FINAL_TRACK_DIR/f"{dt:%Y%m%d}.npz"
    if not p.exists():
        continue

    tr=load_track(p)
    b1=branch1_from_track(tr["lon"],tr["lat"])
    b2=branch2_from_track(tr["lon"],tr["lat"])

    if (
        b1 is not None and b2 is not None and
        np.isfinite(b1).all() and np.isfinite(b2).all()
    ):
        D.append(dt); B1.append(b1); B2.append(b2); SRC.append(r.source)

if not D:
    raise RuntimeError("No valid dual-branch paths")

B1=np.stack(B1)
B2=np.stack(B2)

dataset_path=DATASET_DIR/"daily_dual_branch_paths_FIX6.npz"
np.savez_compressed(
    dataset_path,
    dates=np.array([d.strftime("%Y-%m-%d") for d in D]),
    branch1_lon=B1,
    branch1_target_lats=TARGET_LATS_1,
    branch2_lat=B2,
    branch2_target_lons=TARGET_LONS_2,
    source=np.asarray(SRC)
)

pd.DataFrame({
    "date":D,
    "source":SRC
}).to_csv(
    DATASET_DIR/"daily_dual_branch_meta_FIX6.csv",
    index=False,
    encoding="utf-8-sig"
)

print("dataset:",dataset_path)
print("dual-branch days:",len(D))


In [ ]:

# FIX6-FIX1: WSL重启后从磁盘恢复dataset日期
dataset_path = DATASET_DIR/"daily_dual_branch_paths_FIX6.npz"
if not dataset_path.exists():
    raise FileNotFoundError(dataset_path)
with np.load(dataset_path) as _ds:
    D = list(pd.to_datetime(_ds["dates"]))


# ===== 10. R2 anchor readiness：提前判断180/120/90/60/30天是否可用 =====
avail=pd.Series(False,index=pd.date_range(min(D),max(D),freq="D"))
avail.loc[pd.DatetimeIndex(D)]=True

def longest_missing_run(mask):
    best=cur=0
    for v in mask:
        if bool(v):
            cur+=1; best=max(best,cur)
        else:
            cur=0
    return best

def classify_target_dates(tdates):
    tr_end=pd.Timestamp(cfg.R2_TRAIN_TARGET_END)
    va_end=pd.Timestamp(cfg.R2_VAL_TARGET_END)
    tmin,tmax=min(tdates),max(tdates)
    if tmax<=tr_end: return "train"
    if tmin>tr_end and tmax<=va_end: return "val"
    if tmin>va_end: return "test"
    return "gap"

readiness=[]

for lb in cfg.R2_LOOKBACKS:
    counts={"train":0,"val":0,"test":0,"gap":0}
    total=0

    min_issue=avail.index.min()+pd.Timedelta(days=lb-1)
    max_issue=avail.index.max()-pd.Timedelta(days=max(cfg.R2_LEADS))

    if min_issue<=max_issue:
        for issue in pd.date_range(min_issue,max_issue,freq="D"):
            win=avail.loc[issue-pd.Timedelta(days=lb-1):issue]
            missing=(~win.to_numpy())

            if missing.mean()>cfg.R2_MAX_INPUT_MISSING_FRAC:
                continue
            if longest_missing_run(missing)>cfg.R2_MAX_INPUT_GAP:
                continue

            tdates=[issue+pd.Timedelta(days=x) for x in cfg.R2_LEADS]
            if any(t not in avail.index or not bool(avail.loc[t]) for t in tdates):
                continue

            s=classify_target_dates(tdates)
            counts[s]+=1
            total+=1

    readiness.append({
        "lookback_days":lb,
        "anchors_total":total,
        **counts
    })

readiness_df=pd.DataFrame(readiness)
readiness_df.to_csv(
    REPORT_DIR/"R2_anchor_readiness_FIX6.csv",
    index=False,
    encoding="utf-8-sig"
)
display(readiness_df)


In [ ]:

# FIX6-FIX1: post-crash reload
base_report = pd.read_csv(BASE_REPORT_PATH, parse_dates=["date"])
final_report = pd.read_csv(
    REPORT_DIR/"final_tracker_report_FIX6.csv",
    parse_dates=["date"]
)
recoveries = pd.read_csv(
    REPORT_DIR/"recovery_summary_FIX6.csv",
    parse_dates=["date"]
)
manual_path = REPORT_DIR/"manual_review_required_FIX6.csv"
manual = (
    pd.read_csv(manual_path, parse_dates=["date"])
    if manual_path.exists()
    else pd.DataFrame()
)
readiness_df = pd.read_csv(
    REPORT_DIR/"R2_anchor_readiness_FIX6.csv"
)
dataset_path = DATASET_DIR/"daily_dual_branch_paths_FIX6.npz"
with np.load(dataset_path) as _ds:
    D = list(pd.to_datetime(_ds["dates"]))


# ===== 11. Handoff =====
handoff={
    "tracker":"R1_FIX6_BASELINE_FIRST_FAILURE_ONLY_AUTORETRY",
    "baseline_days":int(len(base_report)),
    "baseline_completed":int(base_report.completed.sum()),
    "baseline_completion":float(base_report.completed.mean()),
    "fix6_completed":int(final_report.completed.sum()),
    "fix6_completion":float(final_report.completed.mean()),
    "added_completed_days":int(final_report.completed.sum()-base_report.completed.sum()),
    "baseline_success_regressions":0,
    "failed_days":int((~base_report.completed).sum()),
    "failed_days_with_any_completed_retry":int(recoveries.recovered.sum()) if len(recoveries) else 0,
    "auto_accepted_recoveries":int(recoveries.auto_accepted.sum()) if len(recoveries) else 0,
    "manual_review_required":int(len(manual)),
    "dual_branch_days":int(len(D)),
    "dataset":str(dataset_path),
    "r2_readiness":readiness_df.to_dict(orient="records"),
}

with open(REPORT_DIR/"R1_FIX6_HANDOFF.json","w",encoding="utf-8") as f:
    json.dump(handoff,f,indent=2,ensure_ascii=False)

print(json.dumps(handoff,indent=2,ensure_ascii=False))



## 如果旧 Cell 8 已经把 WSL 弄崩

不用重跑 recovery。

Cell 8 崩溃前，Cell 6/7 已经把核心结果写入：

```text
R1_FIX6/reports/recovery_summary_FIX6.csv
R1_FIX6/reports/final_tracker_report_FIX6.csv
R1_FIX6/tracks_final/
```

重新打开 WSL 后：

- 运行配置/import/函数定义单元；
- 可以直接运行新的 Cell 8；
- 或者时间很紧时**直接跳过 Cell 8**；
- 继续 Cell 9 → 10 → 11。

QC figure 不参与任何数值计算。
